# 00 — Environment check: CUDA

Verifies the device, loads ModernBERT-large, and runs forward + backward + optimizer steps with LoRA and with full fine-tuning at training shape. Reports peak memory.

All logic lives in `jevlite.envcheck`; this notebook only differs from its twin in the first code cell.

## 1. Environment and config

The only stack-specific cell. Env vars are applied before torch is imported.

In [ ]:
STACK = "cuda"

from jevlite.config import apply_env, load_config, seed_everything
from jevlite.log import logger, setup_logging

setup_logging()
cfg = load_config(STACK)
apply_env(cfg)
logger.info("stack {}: config loaded", STACK)

## 2. Device report

In [ ]:
import pprint

import torch
import transformers

from jevlite.device import device_report, resolve_runtime

rt = resolve_runtime(cfg)
report = device_report(rt) | {"transformers": transformers.__version__}
logger.info("device report:\n{}", pprint.pformat(report))
_ = seed_everything(cfg["seed"])

## 3. Request smoke test

Sample request → schema → encoding → collate → encoder → grouped logits, on the device. The probe head is random, so only shapes and masking are checked.

In [ ]:
from jevlite.envcheck import request_forward_check

smoke = request_forward_check(cfg, rt)
logger.info("request smoke test:\n{}", pprint.pformat(smoke))
assert smoke["ok"]

## 4. Training step: LoRA and full fine-tuning

Uses `train.per_device_batch_size` and `encoding.max_len` from the config, the configured precision (AMP + GradScaler where applicable) and optimizer.

In [ ]:
from jevlite.envcheck import train_step_check

batch_size = cfg["train"]["per_device_batch_size"]
seq_len = cfg["encoding"]["max_len"]
results = []
for mode in ("lora", "full"):
    r = train_step_check(cfg, rt, mode=mode, batch_size=batch_size, seq_len=seq_len)
    logger.info("{} training step:\n{}", mode, pprint.pformat(r))
    assert r["params_updated"], f"{mode}: parameters did not change"
    results.append(r)

## 5. Summary

In [ ]:
cols = ["mode", "precision", "optimizer", "batch_size", "seq_len", "trainable_pct", "step_s", "peak_memory_gb"]
lines = [" | ".join(cols)] + [" | ".join(str(r[c]) for c in cols) for r in results]
logger.info("summary:\n{}", "\n".join(lines))
logger.info("device: {}, default tuning mode for this stack: {}", report["device"], cfg["tuning"]["mode"])